In [11]:
import mlflow
from mlflow.tracking import MlflowClient

tracking_uri = "http://127.0.0.1:5000"
experiment_name = "nyc-taxi-duration"
registered_model_name = "nyc-taxi-duration-model"

mlflow.set_tracking_uri(tracking_uri)
client = MlflowClient(tracking_uri=tracking_uri)

In [12]:
experiments = client.search_experiments()

for experiment in experiments:
    print(experiment.experiment_id, experiment.name)


1 nyc-taxi-duration
0 Default


In [13]:
experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment is None:
    raise ValueError(f"Expérience introuvable : {experiment_name}")

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.validation_rmse ASC"],
    max_results=5,
)

if runs.empty:
    raise ValueError("Aucune exécution terminée n'a été trouvée")

best_run = runs.iloc[0]
runs[["run_id", "params.model", "metrics.validation_rmse"]]


ValueError: Aucune exécution terminée n'a été trouvée

In [14]:
run_id = best_run["run_id"]
model_uri = f"runs:/{run_id}/model"

model_version = mlflow.register_model(
    model_uri=model_uri,
    name=registered_model_name,
)

model_version.version


NameError: name 'best_run' is not defined

In [ ]:
versions = client.search_model_versions(
    filter_string=f"name = '{registered_model_name}'"
)

for item in versions:
    print(
        "version:", item.version,
        "run:", item.run_id,
        "status:", item.status,
    )


In [ ]:
version = model_version.version

client.update_model_version(
    name=registered_model_name,
    version=version,
    description="Modèle candidat pour la prédiction de durée des trajets Yellow Taxi.",
)

client.set_model_version_tag(
    name=registered_model_name,
    version=version,
    key="validation_status",
    value="reviewed",
)


In [ ]:
client.set_registered_model_alias(
    name=registered_model_name,
    alias="champion",
    version=version,
)


In [ ]:
champion_version = client.get_model_version_by_alias(
    name=registered_model_name,
    alias="champion",
)

champion_version.version, champion_version.run_id


In [ ]:
champion_uri = f"models:/{registered_model_name}@champion"
champion_model = mlflow.sklearn.load_model(champion_uri)

champion_predictions = champion_model.predict(X_val[:5])
champion_predictions


In [ ]:
registered_models = client.search_registered_models()

for model in registered_models:
    print(model.name)


In [ ]:
download_path = client.download_artifacts(
    run_id=run_id,
    path="model",
    dst_path="downloaded-artifacts",
)

download_path
